### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [3]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [4]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [7]:
import sys
!{sys.executable} -m pip install grpcio grpcio-tools

   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   --------------------- ------------------ 2.6/4.9 MB 25.6 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 29.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 23.6 MB/s  0:00:00

  Attempting uninstall: protobuf

    Found existing installation: protobuf 5.29.6

    Uninstalling protobuf-5.29.6:

      Successfully uninstalled protobuf-5.29.6

   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobuf]
   ---------------------------------------- 0/3 [protobu

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [9]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [10]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [11]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [12]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [13]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [14]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some key pros of using AutoGen in your new AI Agent project:

1. **Modularity and Customization**: AutoGen is designed to be modular, allowing developers to customize interactions and behaviors of individual agents to fit specific project needs.

2. **Multi-Agent Collaboration**: The framework excels in multi-agent setups, enhancing problem-solving capabilities through collaborative efforts, which can lead to more effective outcomes in complex scenarios.

3. **Scalability**: AutoGen can easily scale its operations, making it suitable for both small and large projects. It can handle multiple tasks efficiently, adapting to the scale of the project.

4. **Automation of Complex Workflows**: It is particularly effective for automating intricate processes like multi-step data analysis and document processing, improving operational efficiency.

5. **Ease of Maintenance**: The framework is designed for easy maintenance, which helps in reducing the overhead associated with managing complex AI systems.

6. **Enhanced AI Capabilities**: AutoGen provides advanced capabilities suited for dynamic environments, making it an excellent choice for projects that require innovative AI solutions.

By choosing AutoGen, you can leverage these benefits to create a more efficient, scalable, and effective AI agent system. 

TERMINATE

## Cons of AutoGen:
Here are some cons associated with using AutoGen in an AI Agent project:

1. **Learning Curve**: AutoGen has a steep learning curve, requiring users to invest time in understanding its features and functionalities, which can be a significant barrier for new users.

2. **Production Limitations**: It is primarily described as a research prototype by Microsoft, making it less suitable for production-level applications. This may impact the reliability and robustness required for commercial use.

3. **Complexity Management**: As projects grow in complexity, managing multiple nested chains and workflows in AutoGen can become cumbersome, leading to increased maintenance challenges and potential for errors.

4. **Cost of Multi-Agent Workflows**: Implementing and managing workflows involving multiple agents can be expensive, which may not be ideal for smaller projects or teams with limited budgets.

5. **Lack of Visual Tools**: AutoGen does not offer a visual builder or no-code editor, making it less accessible for non-technical users who may struggle with more complex programming requirements.

These cons suggest that while AutoGen has its strengths, it may not be the best fit for all projects, particularly those that require ease of use, straightforward deployments, or a low barrier to entry for users with varying technical skills.

TERMINATE



## Decision:

Based on the research provided, I recommend **not using AutoGen** for the project. While it offers significant advantages in modularity, scalability, and multi-agent collaboration, the cons related to its steep learning curve, production limitations, and complexity management present considerable risks. The fact that it is primarily a research prototype makes it less reliable for commercial applications, and the challenges in managing workflows as the project scales could hinder productivity. Therefore, it may be prudent to explore alternatives that offer a more user-friendly experience and proven production viability.

TERMINATE

In [15]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [16]:
await host.stop()